# Predictive Analytics Using Historical Data
### Forecast future sales trends using machine learning

**Objective:** Clean historical sales data, engineer time-based features, train a predictive model, evaluate its accuracy, and forecast future sales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('historical_sales_data.csv', parse_dates=['Date'])
df.head()

## 1. Data Cleaning and Exploration

In [ ]:
df = df.sort_values('Date').reset_index(drop=True)
print('Shape:', df.shape)
print('\nMissing values before cleaning:')
print(df.isnull().sum())

df['Sales'] = df['Sales'].interpolate().bfill().ffill()
print('\nMissing values after cleaning:')
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(df['Date'], df['Sales'])
plt.title('Historical Monthly Sales Trend')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(alpha=0.25)
plt.show()

## 2. Feature Engineering

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['TimeIndex'] = np.arange(len(df))

for lag in [1, 3, 6, 12]:
    df[f'Lag_{lag}'] = df['Sales'].shift(lag)

df['RollingMean_3'] = df['Sales'].rolling(3).mean()
df['RollingMean_6'] = df['Sales'].rolling(6).mean()
df = df.dropna().reset_index(drop=True)

features = ['Year','Month','TimeIndex','Promotion','Lag_1','Lag_3','Lag_6','Lag_12','RollingMean_3','RollingMean_6']
df[features + ['Sales']].head()

## 3. Train-Test Split

In [ ]:
split = int(len(df) * 0.80)
train = df.iloc[:split]
test = df.iloc[split:]
print('Training rows:', len(train))
print('Testing rows:', len(test))

## 4. Train Random Forest Regression Model

In [ ]:
model = RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=2, random_state=42)
model.fit(train[features], train['Sales'])
predictions = model.predict(test[features])

mae = mean_absolute_error(test['Sales'], predictions)
rmse = np.sqrt(mean_squared_error(test['Sales'], predictions))
r2 = r2_score(test['Sales'], predictions)
mape = np.mean(np.abs((test['Sales'] - predictions) / test['Sales'])) * 100

print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R2: {r2:.3f}')
print(f'MAPE: {mape:.2f}%')

## 5. Evaluate and Visualize Predictions

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(test['Date'], test['Sales'], label='Actual')
plt.plot(test['Date'], predictions, label='Predicted')
plt.title('Actual vs Predicted Sales')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Forecast the Next 6 Months

In [ ]:
# Recursive six-month forecast. Future promotions are assumed to be 0.
history = df[['Date','Sales','Promotion']].copy()
future = []

for i in range(6):
    next_date = history['Date'].max() + pd.offsets.MonthBegin(1)
    vals = history['Sales'].tolist()
    row = {
        'Year': next_date.year,
        'Month': next_date.month,
        'TimeIndex': len(df) + i,
        'Promotion': 0,
        'Lag_1': vals[-1], 'Lag_3': vals[-3], 'Lag_6': vals[-6], 'Lag_12': vals[-12],
        'RollingMean_3': np.mean(vals[-3:]), 'RollingMean_6': np.mean(vals[-6:])
    }
    forecast = model.predict(pd.DataFrame([row])[features])[0]
    future.append({'Date': next_date, 'ForecastSales': forecast})
    history = pd.concat([history, pd.DataFrame([{'Date': next_date, 'Sales': forecast, 'Promotion': 0}])], ignore_index=True)

future_df = pd.DataFrame(future)
display(future_df)

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(df['Date'], df['Sales'], label='Historical')
plt.plot(future_df['Date'], future_df['ForecastSales'], marker='o', label='6-Month Forecast')
plt.title('Future Sales Forecast')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 7. Conclusion
The model uses historical sales, time-based features, lag values, rolling averages, and promotion information to forecast future sales. Model performance is evaluated using MAE, RMSE, R², and MAPE. The six-month forecast can support inventory planning, marketing decisions, and sales forecasting.

**Important:** The included dataset is synthetic and intended for learning/demo purposes. A real deployment should use validated business data and compare multiple forecasting approaches.